# Importing Libraries

In [72]:
from datasets import load_dataset, get_dataset_split_names, get_dataset_config_names, load_dataset_builder, Dataset
from transformers import AutoTokenizer
from pathlib import Path
import re

# Inspecting Dataset

**Reference**:
_MTS, Department of War UAP Release 1 — structured corpus, 2026. CC-BY-4.0._
This dataset is a structured, machine-readable companion to the source material at [war.gov/UFO/](https://www.war.gov/UFO/).

In [2]:
ufo_dataset = "MTSLIVE/war-gov-uap-release-1"
get_dataset_config_names(ufo_dataset)   # dataset files

['documents', 'pages', 'figures', 'videos']

We will use the `pages` file.

In [3]:
ds_ufo_builder = load_dataset_builder(ufo_dataset, "pages")
ds_ufo_builder.info.features

{'document_id': Value('string'),
 'page_no': Value('int64'),
 'text': Value('string'),
 'has_figures': Value('bool')}

In [4]:
pages = load_dataset(ufo_dataset, "pages", split="train")
pages

Dataset({
    features: ['document_id', 'page_no', 'text', 'has_figures'],
    num_rows: 4239
})

In [5]:
# example of what we are going to use
print(pages[0]['text'])

HEADQUARTERS
AIR MATERIEL COMMAND
WRIGHT FIELD, DAYTON, OHIO

DEC 1 9 1947

SUBJECT: Flying Discs

TO: Chief of Staff
United States Air Force
Washington 25, D. C.
ATTENTION: Director, Research & Development
Major General L. C. Craigie

1. Confirming the recent conversation of the undersigned with Major General L. C. Craigie, 9 December 1947, attached as listed below are copies of the reports from this Headquarters concerning Flying Discs.

2. Comments of Headquarters, Air Force on these letters have never been received by this Command. Continued and recent reports from qualified observers concerning this phenomenon still makes this matter one of concern to Headquarters, Air Materiel Command. Intelligence Department of this Command is continuing the collection and analysis of all available reports.

FOR THE COMMANDING GENERAL:

H. M. McCOY
Colonel, USAF
Chief of Intelligence

2 Attach:
cc ltr to CG, AAF, dtd 23 Sept 47 subj "AMC Opinion Concerning "Flying Discs""
cc ltr to CG, AAF, dtd 

## Check some documents by id

In [6]:
# all documents, each of these is composed of 1 or more pages
IDS = set(pages["document_id"])

Let's see if some pages have less than 20 characters.

In [38]:
null_id = 0     # count for document_id (even if it's just one page)
null_pgs = 0    # count for pages (can share the same id)
doc_ids = []    # to filter later(?)

for page in pages:
    if len(page["text"]) <= 20:
        null_pgs+=1
        if page["document_id"] not in doc_ids:
            null_id+=1
            doc_ids.append(page["document_id"])
        #print(f"doc_id: {page["document_id"]}\n text: {page["text"]}", "\n")

print(f"total null ids (at least one page): {null_id}")
print(f"total null pages: {null_pgs}")
print(f"problematic ids: {doc_ids}")

total null ids (at least one page): 61
total null pages: 225
problematic ids: ['342-hs1-416511228-319-1-flying-discs-1949', '65-hs1-101634279-100-de-26505', '65-hs1-834228961-62-hq-83894-section-1', '65-hs1-834228961-62-hq-83894-section-10', '65-hs1-834228961-62-hq-83894-section-2', '65-hs1-834228961-62-hq-83894-section-3', '65-hs1-834228961-62-hq-83894-section-4', '65-hs1-834228961-62-hq-83894-section-5', '65-hs1-834228961-62-hq-83894-section-6', '65-hs1-834228961-62-hq-83894-section-7', '65-hs1-834228961-62-hq-83894-section-8', '65-hs1-834228961-62-hq-83894-section-9', '65-hs1-834228961-62-hq-83894-serial-130', '65-hs1-834228961-62-hq-83894-serial-164', '65-hs1-834228961-62-hq-83894-serial-438', 'dow-uap-d10-mission-report-middle-east-may-2022', 'dow-uap-d12-mission-report-iraq-may-2022', 'dow-uap-d25-mission-report-greece-january-2024', 'dow-uap-d27-mission-report-united-arab-emirates-october-2023', 'dow-uap-d28-mission-report-iraq-september-2024', 'dow-uap-d3-mission-report-arabian

So there are 61 `document_id` with _at least_ one page with less than 20 characters. If we talk in terms of pages, there are 225 pages almost empty.

In [ ]:
count = pages.to_pandas().groupby("document_id").count()
single = list(count[count["page_no"]==1].index)  # Documents of 1 page only
# Print the single paged documents
for doc in pages.filter(lambda x: x["document_id"] in single): print(f"DOCUMENT: {doc["document_id"].upper()}\n\n{doc["text"]}\n\n\n\n")

In [7]:
# Discard single paged documents containing no information
pattern = re.compile("fbi-photo|nasa-uap-vm|fbi-september-2023-sighting-composite-sketch")  # compile pattern to match discarded documents
useIDS = set(ID for ID in IDS if not pattern.match(ID))  # discard matches

# Join pages of same documents

In [29]:
# Example of a joined document
docs = pages.filter(lambda x: x["document_id"] == list(useIDS)[0])
text = "\n------------------\n".join(docs["text"])
print(text)

FEDERAL BUREAU OF INVESTIGATION

Date of entry 10/ /2023

On September 2023, and and FBI Special Agent interviewed via Facetime video. a with sat in on the interview ( and were in at the time of the interview). After being advised of the identity of the interviewing agents and the nature of the interview, provided the following information:

was a in the of He'd been an since and had ten to fifteen hours as a drone pilot.

On September 2023 was at with three other contractors and for LiDAR tests with After receiving a the five of them got into three vehicles with driving the first vehicle and as her passenger.

The three vehicles began driving south around 7:30 am. The sun was in the east with good visibility.

The three vehicles came to a gate that was giving trouble. At about three quarters of the windshield up, saw a linear object with a super bright light on the east side of the object. The light was bright white and bright enough to see bands within the light. The object was metal

In [8]:
# Create new dataset with joined pages
newDs = {"document_id": [], "text": []}
for ID in useIDS:
    entries = pages.filter(lambda x: x["document_id"] == ID)
    newDs["document_id"].append(ID)
    newDs["text"].append("\n".join(entries["text"]))
newDs = Dataset.from_dict(newDs)

In [24]:
newDs[0]["text"]

'On 2025, at approximately 1700 hours [WITNESS 1 (a senior US intelligence official)], [FEDERAL PARTNER 1] and [FEDERAL PARTNER 2], accompanied by [WITNESS 2 (a senior US intelligence official)], and two pilots from [STATE PARTNER ORGANIZATION], departed the [OPERATIONS CENTER] ([COORDINATES]), ([FACILITY]) main area via a [STATE PARTNER ORGANIZATION] helicopter (call sign [CALL SIGN 1]) to conduct a daytime ariel search of the [MOUNTAIN RANGE NAME] west of [SITE CODE NAME] on [FACILITY]. Previous eyewitness reports from personnel who observed orbs/lights IVO ([COORDINATES]) cited hearing thuds as if something has fallen and hit the ground. (Note: Earlier that day the Office completed a successful test of the at [SITE CODE NAME] ([COORDINATES]) on [FACILITY].\n\nAt 1751 hours [CALL SIGN 1] spotted a large cavern entrance ([COORDINATES]) and conducted a short orbit of the location\n\nAt approximately 2050 hours [CALL SIGN 1] landed west of the mountains and dropped off [WITNESS 2] with 

In [79]:
Path("models_cache").mkdir(exist_ok=True)
tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-uncased", cache_dir="models_cache") 
toks, length = tokenizer_bert(newDs[0]["text"], truncation=True, return_attention_mask=False, return_length=True, return_token_type_ids=False, add_special_tokens=False, max_length=int(1e18)).values()
print(length[0], toks[:200])  # sadly length is a list

1638 [2006, 16798, 2629, 1010, 2012, 3155, 16601, 2847, 1031, 7409, 1015, 1006, 1037, 3026, 2149, 4454, 2880, 1007, 1033, 1010, 1031, 2976, 4256, 1015, 1033, 1998, 1031, 2976, 4256, 1016, 1033, 1010, 5642, 2011, 1031, 7409, 1016, 1006, 1037, 3026, 2149, 4454, 2880, 1007, 1033, 1010, 1998, 2048, 8221, 2013, 1031, 2110, 4256, 3029, 1033, 1010, 7745, 1996, 1031, 3136, 2415, 1033, 1006, 1031, 12093, 1033, 1007, 1010, 1006, 1031, 4322, 1033, 1007, 2364, 2181, 3081, 1037, 1031, 2110, 4256, 3029, 1033, 7739, 1006, 2655, 3696, 1031, 2655, 3696, 1015, 1033, 1007, 2000, 6204, 1037, 12217, 16126, 3945, 1997, 1996, 1031, 3137, 2846, 2171, 1033, 2225, 1997, 1031, 2609, 3642, 2171, 1033, 2006, 1031, 4322, 1033, 1012, 3025, 3239, 9148, 27401, 4311, 2013, 5073, 2040, 5159, 19607, 2015, 1013, 4597, 28346, 1006, 1031, 12093, 1033, 1007, 6563, 4994, 20605, 2015, 2004, 2065, 2242, 2038, 5357, 1998, 2718, 1996, 2598, 1012, 1006, 3602, 1024, 3041, 2008, 2154, 1996, 2436, 2949, 1037, 3144, 3231, 1997, 1996, 